In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# train_slice_interp.py
import os
import random
from typing import Tuple
import re
import math
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
import torchvision.transforms.functional as TF
from safetensors.torch import save_file, load_file
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from accelerate import Accelerator

# diffusers imports
from diffusers import DDPMScheduler, UNet2DConditionModel, AutoencoderKL

import matplotlib.pyplot as plt

def denorm(x):
    """[-1, 1] -> [0, 1]"""
    return (x.clamp(-1, 1) + 1) / 2


def plot_triplet(img_prev, img_mid, img_next, idx=0):
    """
    img_prev, img_mid, img_next: [B, 3, H, W]
    idx: which sample in the batch to visualize
    """
    imgs = [img_prev, img_mid, img_next]
    titles = ["prev", "mid (gt)", "next"]

    plt.figure(figsize=(12, 4))
    for i, (img, title) in enumerate(zip(imgs, titles)):
        x = img[idx].detach().cpu()      # [3, H, W]
        x = denorm(x)
        x = x.permute(1, 2, 0)           # -> [H, W, 3]

        plt.subplot(1, 3, i + 1)
        plt.imshow(x)
        plt.title(title)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

class Transpose(nn.Module):
    def __init__(self, dim0, dim1):
        super().__init__()
        self.dim0 = dim0
        self.dim1 = dim1

    def forward(self, x):
        return x.transpose(self.dim0, self.dim1)

class PositionalEncoding2D(nn.Module):
    def __init__(self, num_patches, dim):
        super().__init__()
        self.register_buffer('pos_embed', self.build_sincos_encoding(num_patches, dim), persistent=False)

    def build_sincos_encoding(self, num_patches, dim):
        pe = torch.zeros(num_patches, dim)
        position = torch.arange(0, num_patches, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # [1, num_patches, dim]

    def forward(self, x):
        return x + self.pos_embed[:, :x.size(1), :]

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels)
        )
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return self.block(x) + self.skip(x)

class CondEncoder(nn.Module):
    def __init__(self, in_channels=4, out_channels=768, num_tokens=64):
        super().__init__()
        self.encoder = nn.Sequential(
            ResidualBlock(in_channels, 64), # [B, 64, 64, 64]
            nn.AvgPool2d(2), # [B, 64, 32, 32]
            ResidualBlock(64, 128),
            nn.AvgPool2d(2), # [B, 128, 16, 16]
            ResidualBlock(128, 256),
            nn.AvgPool2d(2), # [B, 256, 8, 8]
            nn.Conv2d(256, out_channels, kernel_size=1) # [B, 736, 8, 8]
        )
        self.proj = nn.Sequential(
            nn.Flatten(2),  # [B, 736, 64]
            Transpose(-1, -2),   # [B, 64, 736]
        )
        self.pos_embed = PositionalEncoding2D(num_patches=num_tokens, dim=out_channels)
        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x):
        feat = self.encoder(x)          # [B, 736, 8, 8]
        tokens = self.proj(feat)        # [B, 64, 736]
        tokens = self.pos_embed(tokens) # [B, 64, 736]
        tokens = self.norm(tokens)
        return tokens


# -------------------------
# Dataset: neighbouring slices
# -------------------------

class Slice3DDataset(Dataset):
    """
    Filename format: <z>_<region>.png

    - prefix  z       : slice index along z-axis
    - suffix  region  : patch / region id (must stay the same)

    For each fixed region:
        choose endpoints (z_i, z_j), j >= i+2
        for every z_t where i < t < j:
            input  : (z_i, z_j)
            target : z_t
            weights: (w_prev, w_next), sum to 1
    """

    def __init__(self, root_dir):
        self.root_dir = root_dir

        self.to_tensor = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5],
                                 [0.5, 0.5, 0.5])
        ])

        # ------------------------------------------
        # Parse filenames: z_region.png
        # ------------------------------------------
        pattern = re.compile(r"^(\d+)_(\d+)\.png$")

        parsed = []  # (z, region, path)
        for fname in os.listdir(root_dir):
            m = pattern.match(fname)
            if m is None:
                continue
            z = int(m.group(1))        # prefix = z index
            region = int(m.group(2))   # suffix = region id
            parsed.append((z, region, os.path.join(root_dir, fname)))

        if len(parsed) == 0:
            raise RuntimeError("No valid <z>_<region>.png files found.")

        # ------------------------------------------
        # Group by region (suffix!)
        # ------------------------------------------
        region_groups = {}
        for z, region, path in parsed:
            region_groups.setdefault(region, []).append((z, path))

        # ------------------------------------------
        # Build interpolation samples
        # ------------------------------------------
        self.base_samples = []
        # each: (prev_path, next_path, gt_path, w_prev, w_next)

        for region, items in region_groups.items():
            items = sorted(items, key=lambda x: x[0])  # sort by z
            n = len(items)
            if n < 3:
                continue

            for i in range(n-2):
                z_i, path_i = items[i]
                for j in range(i + 2, n):
                    z_j, path_j = items[j]
                    denom = z_j - z_i
                    for t in range(i + 1, j):
                        z_t, path_t = items[t]

                        d_prev = z_t - z_i
                        d_next = z_j - z_t

                        w_prev = d_next / (d_prev + d_next)
                        w_next = d_prev / (d_prev + d_next)

                        self.base_samples.append(
                            (path_i, path_j, path_t,
                             float(w_prev), float(w_next))
                        )

        if len(self.base_samples) == 0:
            raise RuntimeError("No valid interpolation samples constructed.")

        # ------------------------------------------
        # deterministic augmentations
        # ------------------------------------------
        self.num_rotations = 4
        self.num_hflip = 2
        self.num_vflip = 2
        self.num_aug = self.num_rotations * self.num_hflip * self.num_vflip

    def __len__(self):
        return len(self.base_samples) * self.num_aug

    def __getitem__(self, idx):
        base_idx = idx // self.num_aug
        rem = idx % self.num_aug

        rot_k = rem // (self.num_hflip * self.num_vflip)
        rem %= (self.num_hflip * self.num_vflip)
        hflip = rem // self.num_vflip
        vflip = rem % self.num_vflip

        prev_path, next_path, gt_path, w_prev, w_next = \
            self.base_samples[base_idx]

        img_prev = Image.open(prev_path).convert("RGB")
        img_next = Image.open(next_path).convert("RGB")
        img_gt   = Image.open(gt_path).convert("RGB")

        if rot_k > 0:
            angle = 90 * rot_k
            img_prev = img_prev.rotate(angle)
            img_next = img_next.rotate(angle)
            img_gt   = img_gt.rotate(angle)

        if hflip:
            img_prev = img_prev.transpose(Image.FLIP_LEFT_RIGHT)
            img_next = img_next.transpose(Image.FLIP_LEFT_RIGHT)
            img_gt   = img_gt.transpose(Image.FLIP_LEFT_RIGHT)

        if vflip:
            img_prev = img_prev.transpose(Image.FLIP_TOP_BOTTOM)
            img_next = img_next.transpose(Image.FLIP_TOP_BOTTOM)
            img_gt   = img_gt.transpose(Image.FLIP_TOP_BOTTOM)

        return (
            self.to_tensor(img_prev),
            self.to_tensor(img_next),
            self.to_tensor(img_gt),
            torch.tensor(w_prev, dtype=torch.float32),
            torch.tensor(w_next, dtype=torch.float32)
        )


# -------------------------
# Trainer
# -------------------------
class Slice3DTrainer:
    def __init__(
        self,
        train_dir,
        val_dir,
        pretrained_path="runwayml/stable-diffusion-v1-5",
        batch_size=8,
        val_batch_size=8,
        img_size=512,
        num_tokens=64,
        cond_dim=768,
        lr=2e-5,
    ):
        self.accelerator = Accelerator(mixed_precision="fp16")
        self.device = self.accelerator.device

        # datasets & dataloaders
        train_ds = Slice3DDataset(train_dir)
        val_ds = Slice3DDataset(val_dir)

        self.train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
        self.val_loader   = DataLoader(val_ds, batch_size=val_batch_size, shuffle=False, num_workers=2, pin_memory=True)

        # load pretrained vae and unet
        print("Loading VAE and UNet from:", pretrained_path)
        self.vae = AutoencoderKL.from_pretrained(pretrained_path, subfolder="vae")
        self.unet = UNet2DConditionModel.from_pretrained(pretrained_path, subfolder="unet")
        self.cond_proj = CondEncoder()
        state = load_file("drive/MyDrive/checkpoints_3d/unet.safetensors")
        missing, unexpected = self.unet.load_state_dict(state, strict=True)
        state = load_file("drive/MyDrive/checkpoints_3d/condencoder.safetensors")
        missing, unexpected = self.cond_proj.load_state_dict(state, strict=True)

        # Slight safety: if vae has attribute scaling_factor; otherwise use 0.18215 default from SD
        self.scaling_factor = getattr(self.vae.config, "scaling_factor", 0.18215)

        # conditional encoder: it expects concatenated channels: we'll assume
        # VAE latent channels are 4 -> two inputs -> 8 channels input to CondEncoder

        # We do not train VAE
        self.vae.requires_grad_(False)
        self.vae.eval()

        # prepare components with accelerator
        components = [self.vae, self.unet, self.cond_proj, self.train_loader, self.val_loader]
        self.vae, self.unet, self.cond_proj, self.train_loader, self.val_loader = \
            self.accelerator.prepare(*components)

        # optimizer for trainable parts (UNet + cond_proj)
        self.optimizer = torch.optim.AdamW(
            list(self.unet.parameters()) + list(self.cond_proj.parameters()),
            lr=lr
        )
        self.optimizer = self.accelerator.prepare(self.optimizer)

        # scheduler
        self.noise_scheduler = DDPMScheduler.from_pretrained(pretrained_path, subfolder="scheduler")
        # keep prediction_type as-is; we operate on model.sample output to match original pattern
        self.noise_scheduler.config.prediction_type = "sample"

        # training bookkeeping
        self.loss_history = []
        self.val_loss_history = []

        # checkpoint registration
        self.accelerator.register_for_checkpointing(self.unet, self.cond_proj, self.optimizer)

    def load_partial_checkpoint(self, ckpt_dir):

       unet_path = os.path.join(ckpt_dir, "unet.safetensors")
       cond_path = os.path.join(ckpt_dir, "condencoder.safetensors")

       # ---- load UNET ----
       if os.path.exists(unet_path):
          print(f"Loading UNET from {unet_path}")
          state = load_file(unet_path)
          missing, unexpected = self.unet.load_state_dict(state, strict=True)
          print("UNET loaded.")
          print("Missing:", missing)
          print("Unexpected:", unexpected)

       # ---- load CondEncoder ----
       if os.path.exists(cond_path):
          print(f"Loading CondEncoder from {cond_path}")
          state = load_file(cond_path)
          missing, unexpected = self.cond_proj.load_state_dict(state, strict=True)
          print("CondEncoder loaded.")
          print("Missing:", missing)
          print("Unexpected:", unexpected)

    def save_best_checkpoint(self, save_dir):
        os.makedirs(save_dir, exist_ok=True)

        # --- UNET ---
        unet_path = os.path.join(save_dir, "unet.safetensors")
        save_file(self.unet.state_dict(), unet_path)
        print(f"[Saved] UNet → {unet_path}")

        # --- CondEncoder ---
        cond_path = os.path.join(save_dir, "condencoder.safetensors")
        save_file(self.cond_proj.state_dict(), cond_path)
        print(f"[Saved] CondEncoder → {cond_path}")

    def train_step(self, batch) -> torch.Tensor:
        """
        batch: (img_prev, img_next, img_mid)
        All tensors are on device (accelerator.prepare DataLoader handles pin).
        Behavior:
          - use prev as the base to add noise
          - use next as the condition (via cond_proj)
          - use mid as the target latent for loss
        """
        img_prev, img_next, img_mid, wp, wn = batch  # each: [B, 3, H, W]

        # encode latents (no grad)
        with torch.no_grad():
            latent_prev = self.vae.encode(img_prev).latent_dist.sample() * self.scaling_factor
            latent_next = self.vae.encode(img_next).latent_dist.sample() * self.scaling_factor
            latent_mid  = self.vae.encode(img_mid).latent_dist.sample() * self.scaling_factor

        # noise + timesteps
        noise = torch.randn_like(latent_prev)
        timesteps = torch.randint(
            0,
            self.noise_scheduler.config.num_train_timesteps,
            (latent_prev.shape[0],),
            device=latent_prev.device,
            dtype=torch.long
        )

        # add noise to the BASE latent (prev) — this is the noisy starting point
        noisy_latents = self.noise_scheduler.add_noise(latent_mid, noise, timesteps)

        # build condition tokens from next latent (use next as condition)
        wp = wp.view(-1, 1, 1, 1)   # [B,1,1,1]
        wn = wn.view(-1, 1, 1, 1)
        condition = self.cond_proj(wp*latent_prev + wn*latent_next)  # [B, num_tokens, cond_dim]

        # predict (model.sample follows previous pattern)
        pred = self.unet(noisy_latents, timesteps, encoder_hidden_states=condition).sample

        # MSE loss between predicted output and target (mid latent)
        # Mid as target, so compare to latent_mid
        loss = F.mse_loss(pred, latent_mid)
        return loss

    def train_step_for_eval(self, batch) -> torch.Tensor:
        """
        Separate wrapper for validation so we keep training semantics same but run without optimizer steps.
        It calls the same logic as train_step but ensures no side-effects.
        """
        with torch.no_grad():
            return self.train_step(batch)

    def validate_step(self):
        self.unet.eval()
        total_loss = 0.0
        n = 0
        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc="Validation"):
                loss = self.train_step(batch)  # train_step uses no grad for VAE encodes; we are in no_grad context
                # ensure scalar
                batch_size = batch[0].shape[0]
                total_loss += loss.item() * batch_size
                n += batch_size
        avg = total_loss / max(1, n)
        self.unet.train()
        return avg

    def _plot_loss_curve(self):
        plt.figure(figsize=(8, 5))
        plt.plot(range(1, len(self.loss_history) + 1), self.loss_history, label="Train")
        plt.plot(range(1, len(self.val_loss_history) + 1), self.val_loss_history, label="Val")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid(True)
        plt.show()
        plt.close()

    def train(self, epochs=100, save_dir="checkpoints", patience=5):
        os.makedirs(save_dir, exist_ok=True)
        best_val = float("inf")
        patience_cnt = 0

        for epoch in range(1, epochs + 1):
            self.unet.train()
            epoch_losses = []
            progress = tqdm(self.train_loader, desc=f"Epoch {epoch} [Train]")
            for batch in progress:
                with self.accelerator.accumulate(self.unet):
                    loss = self.train_step(batch)
                    self.accelerator.backward(loss)
                    if self.accelerator.sync_gradients:
                        self.accelerator.clip_grad_norm_(self.unet.parameters(), 1.5)
                    self.optimizer.step()
                    self.optimizer.zero_grad()
                    epoch_losses.append(loss.detach().cpu().item())
                    progress.set_postfix({"loss": float(np.mean(epoch_losses))})

            epoch_mean = float(np.mean(epoch_losses))
            self.loss_history.append(epoch_mean)

            # validation
            val_loss = self.validate_step()
            self.val_loss_history.append(val_loss)
            print(f"Epoch {epoch} -> Train: {epoch_mean:.6f}  Val: {val_loss:.6f}")

            if val_loss < best_val:
                best_val = val_loss
                patience_cnt = 0
                self.accelerator.wait_for_everyone()
                self.save_best_checkpoint(save_dir)
            else:
                patience_cnt += 1
                print(f"Patience {patience_cnt}/{patience}")

                if patience_cnt >= patience:
                    print("Early stopping.")
                    break


            # optional LR decay every few epochs
            if epoch % 10 == 0:
                for g in self.optimizer.param_groups:
                    g["lr"] *= 0.5
                    print("Decayed lr ->", g["lr"])

            # plot losses
            self._plot_loss_curve()


In [ ]:
# -------------------------
# Main entrypoint
# -------------------------
if __name__ == "__main__":
    # reproducibility
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)

    # Edit these paths as appropriate
    TRAIN_DIR = "drive/MyDrive/hubmap_3d_patches/train"
    VAL_DIR   = "drive/MyDrive/hubmap_3d_patches/val"
    PRETRAIN  = "runwayml/stable-diffusion-v1-5"  # or a local path with subfolders 'vae','unet','scheduler'

    trainer = Slice3DTrainer(
        train_dir=TRAIN_DIR,
        val_dir=VAL_DIR,
        pretrained_path=PRETRAIN,
        batch_size=16,    # reduce if VRAM limited
        val_batch_size=8,
        img_size=512,
        num_tokens=64,
        cond_dim=768,
        lr=1e-5
    )

    trainer.train(epochs=30, save_dir="drive/MyDrive/checkpoints_3d", patience=10)



In [ ]:
trainer.save_best_checkpoint("drive/MyDrive/checkpoints_3d")